[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [asyncpg and psycopg3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)

# A Server of Your Own


## What you will be able to do

Say what psycopg 3 and asyncpg each are, which one to reach for first, and where they sit in
relation to an ORM you may already have used. Have a PostgreSQL server running in your session, from
nothing, in about two minutes, and know which of the four steps each command did. Connect to it from
both drivers over the local socket, with no password anywhere. Say what a cluster, a database, a
role and a connection each are, and which of them the word "database" is being used for in any given
sentence. Show the thing that is worth all of this: two writers at once, which the file on your disk
refused. And read the four ways a connection fails before it ever runs a query.


## The idea

### The two libraries this guide is about

A driver is the piece that speaks PostgreSQL's protocol from Python. It opens the socket, turns your
values into the bytes the server expects, sends statements, and turns the rows that come back into
Python objects. PostgreSQL has no idea Python exists, and the driver is the whole of what stands
between them.

If you have used SQLAlchemy, SQLModel or peewee, you have used a driver without naming one: those
libraries write the SQL and hand it to exactly this layer. This guide is that layer, which is where
you end up when you want `COPY`, `LISTEN`, a JSONB operator or a connection pool you control.

**psycopg 3** is the one most code uses. It follows the DB-API, the same standard interface
`sqlite3` follows, so `connect`, `cursor`, `execute` and `fetchall` mean what you already expect. It
is the broad one: type adaptation you can extend, `COPY`, server-side cursors, pipeline mode and a
pool all come with it, and it works synchronously or asynchronously from the one package.

**asyncpg** is the specialist. It is asynchronous only, it does not follow the DB-API, and it does
not use libpq: it implements PostgreSQL's binary protocol itself, which is where its speed comes
from. It does less, and it is faster at the thing it is for, which is many queries in flight at once.

| | psycopg 3 | asyncpg |
|---|---|---|
| import | `import psycopg` | `import asyncpg` |
| follows the DB-API | yes, `paramstyle` is `pyformat` | no |
| synchronous code | yes | no, every call is awaited |
| asynchronous code | yes, through `AsyncConnection` | yes, and it is the only way |
| placeholder | `%s` | `$1`, `$2` |
| underneath | libpq, PostgreSQL's own C library | its own implementation of the protocol |
| a row arrives as | a tuple, or whatever `row_factory` says | a `Record`, indexed by position or name |

Reach for **psycopg 3** by default. It is the one with the wider feature set, the one whose
interface transfers from `sqlite3`, and the one you can use without rewriting a synchronous program.
Reach for **asyncpg** when the program is already asynchronous and the database is what it spends
its time waiting on, which is the case this guide measures in **Which Driver** rather than asserts.

Plenty of programs use both, and the **An Event Store** notebook does: `COPY` and the bulk writes
through psycopg, the concurrent readers and the listener through asyncpg. That is why this guide
teaches two drivers rather than one, and why every notebook that can show a thing in both does.

### The words, and where each one is explained

PostgreSQL uses several words that a file-based database never needed, and two of them are used for
more than one thing. This table is the map. Every term is either explained in this notebook or
belongs to a later one, and the table says which.

| Term | What it is | Where |
|---|---|---|
| cluster | one running server, with its own port, socket and data directory | here |
| database | one named collection of tables inside a cluster | here |
| role | a user, a group, or both, which is what you connect as | here |
| schema | a named group of tables inside a database, `public` unless you say otherwise | **Connecting and Executing** |
| connection | one session with one database, holding one transaction at a time | here |
| cursor | the thing you run a statement on and read rows back from | **Connecting and Executing** |
| transaction | a run of statements that commit or roll back together | **Transactions and Errors** |
| pool | a set of open connections handed out and given back | **Connection Pools** |
| protocol | the wire format, and the reason a value has to be adapted | **Types and Adaptation** |
| DSN | the connection string, in its several spellings | **Connecting to a Hosted Server** |

The two overloaded words are worth saying plainly now. A **cluster** is the server, and it is not a
group of machines: PostgreSQL's word for what you started, which holds several databases. A
**database** is one of those, not the server, so "the database is down" and "connect to the database"
are about different things.

### The problem

The **sqlite3, Deep Dive** guide's database is a file, and one writer at a time may hold it. A second
writer waits, and then gives up with `database is locked`. That is not a bug and no setting removes
it: a file has no process of its own to arbitrate, so the lock is the whole file.

A server does have a process of its own. It can hold one transaction per connection, take locks per
row rather than per file, and let two writers work on different rows at the same time without either
knowing about the other. That is what this guide is about, and it is what the first look shows.

### What a server is, as far as Python is concerned

A process listening on a socket, speaking a protocol neither driver hides from you completely. The
socket here is a file in a directory, not a network port, which is why nothing in this guide needs a
password: PostgreSQL can see which operating system user opened the socket, and trusts it if a role
of that name exists. That is called peer authentication.

### Why it works that way

Everything else in this guide follows from the server being a separate process. A value has to be
turned into bytes to cross the socket, which is **Types and Adaptation**. A statement costs a round
trip, which is what **Pipeline Mode** and **COPY** are about. A connection is expensive enough to
keep, which is **Connection Pools**. And a failure can arrive from the other side of the socket long
after your code moved on, which is **Transactions and Errors**.

### Where this shows up

The first time two processes write. A web application with more than one worker, a job queue with
more than one consumer, or a script that runs while somebody is using the site.

### What this notebook covers

What the two drivers are and when each one is the right answer. Installing, starting and waiting for
a cluster, one step at a time, since this is the only notebook where that is the subject rather than
the Setup cell. Connecting with both drivers. Two writers at once, against the same pair on SQLite.
Then the four ways a connection fails, which is most of what goes wrong on the first day.

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import sqlite3
import tempfile
from pathlib import Path

import psycopg

path = str(Path(tempfile.mkdtemp()) / "one.db")
first = sqlite3.connect(path, timeout=1)
second = sqlite3.connect(path, timeout=1)
first.execute("CREATE TABLE writers (id INTEGER PRIMARY KEY, who TEXT)")
first.commit()

first.execute("BEGIN IMMEDIATE")                    # one writer, holding the file
first.execute("INSERT INTO writers (who) VALUES ('first')")
try:
    second.execute("INSERT INTO writers (who) VALUES ('second')")
except sqlite3.OperationalError as error:
    print("sqlite3:   ", type(error).__name__ + ":", error)

one = psycopg.connect("dbname=guide")
two = psycopg.connect("dbname=guide")
one.execute("CREATE TABLE IF NOT EXISTS writers (id int PRIMARY KEY, who text)")
one.commit()
one.execute("TRUNCATE writers")
one.commit()

one.execute("INSERT INTO writers VALUES (1, 'first')")      # neither has committed yet
two.execute("INSERT INTO writers VALUES (2, 'second')")
one.commit()
two.commit()
print("PostgreSQL:", one.execute("SELECT id, who FROM writers ORDER BY id").fetchall())
```

```
sqlite3:    OperationalError: database is locked
PostgreSQL: [(1, 'first'), (2, 'second')]
```

The same two writes, twice. The file refused the second one while the first held it. The server took
both, because they were different rows, and neither writer waited for the other. Everything in this
guide is downstream of that difference.


## Setup

Eleven imports, both drivers installed and pinned, a server, and a table with five thousand rows in
it. This cell is the same in every notebook of this guide, and it is the slow one: budget one to
three minutes the first time, and a few seconds after that.

- `psycopg` and `asyncpg` are the two drivers this guide is about
- `subprocess`, `sys`, `os`, `getpass` and `time` install PostgreSQL and wait for it to answer, which
  the worked examples below take apart
- `sqlite3`, `tempfile` and `Path` are for the comparison this notebook ends on
- `version` and `PackageNotFoundError` install the drivers where they are missing

What the cell does, in order: install the drivers; start a server if nothing is answering; make a
database called `guide`; drop any table an earlier run of a notebook left behind; create an `events`
table and put five thousand rows in it; and print one line saying what it is all running against.

That third step is worth knowing about. The `guide` database belongs to this guide, and Setup leaves
it holding exactly one table, so every notebook starts from the same place and running one a second
time gives the same output as running it the first time.

`start_server` is the part worth reading twice. It asks whether anything is answering before it
installs anything, so on a machine that already runs PostgreSQL it does nothing at all. If nothing
answers and the machine is Linux, which is what Colab is, it installs the packages, starts the
cluster, waits for it, and makes a role named after the operating system user, because that is what
peer authentication needs. On any other machine it stops and says so rather than installing
something you did not ask for.


In [1]:
import getpass
import os
import sqlite3
import subprocess
import sys
import tempfile
import time
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    if version("psycopg") < "3.3" or version("asyncpg") < "0.31":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "psycopg[binary,pool]==3.3.6", "psycopg-pool==3.3.2", "asyncpg==0.31.0"],
                   check=True)

import asyncpg
import psycopg

def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(database="postgres"):
    """Whether a server is there, asked the only way that needs no client binaries."""
    try:
        with psycopg.connect(f"dbname={database}", connect_timeout=2):
            return True
    except psycopg.OperationalError:
        return False


def start_server(wait=60):
    """Install and start PostgreSQL if nothing is answering. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No PostgreSQL is answering. Start your own server and run this again: "
                           "this cell only installs one on Linux, which is what Colab runs.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"{sudo}apt-get -qq update")
    shell(f"{sudo}apt-get -qq -y install postgresql postgresql-contrib")
    shell(f"{sudo}service postgresql start")                        # Colab has no systemd

    for attempt in range(1, wait + 1):                              # start returns before it listens
        if shell("pg_isready -q")[0] == 0:
            break
        print(f"  waiting for the cluster ({attempt})")              # a silent minute looks hung
        time.sleep(1)
    else:
        raise RuntimeError(f"PostgreSQL did not accept connections within {wait} seconds.")

    me = getpass.getuser()                                          # peer authentication wants a role
    asking = f"""sudo -u postgres psql -tAc "SELECT 1 FROM pg_roles WHERE rolname='{me}'" """
    if shell(asking)[1] != "1":                                     # named for the operating system user
        shell(f"sudo -u postgres createuser -s {me}")
    return "installed and started"

def build(rows=5000):
    """Make the guide database and its events table, and fill it once."""
    with psycopg.connect("dbname=postgres", autocommit=True) as conn:
        if not conn.execute("SELECT 1 FROM pg_database WHERE datname = 'guide'").fetchone():
            conn.execute("CREATE DATABASE guide")                   # cannot run in a transaction

    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        for (leftover,) in conn.execute(                            # whatever an earlier run made
                "SELECT tablename FROM pg_tables "
                "WHERE schemaname = 'public' AND tablename <> 'events'").fetchall():
            conn.execute(f'DROP TABLE IF EXISTS "{leftover}" CASCADE')

        conn.execute("""CREATE TABLE IF NOT EXISTS events (
                            id bigserial PRIMARY KEY,
                            ts timestamptz NOT NULL DEFAULT now(),
                            kind text NOT NULL,
                            payload jsonb NOT NULL)""")
        if conn.execute("SELECT count(*) FROM events").fetchone()[0] == 0:
            conn.execute("""INSERT INTO events (kind, payload)
                            SELECT (ARRAY['click', 'view', 'purchase'])[1 + n %% 3],
                                   jsonb_build_object('n', n, 'size', 1 + n %% 7)
                            FROM generate_series(1, %s) AS n""", (rows,))
        return conn.execute("SELECT count(*) FROM events").fetchone()[0]

def report():
    """One line naming what this notebook is running against."""
    rows = build()                                                  # makes the database if it is new
    with psycopg.connect("dbname=guide") as conn:
        major = int(conn.execute("SHOW server_version_num").fetchone()[0]) // 10000
    return (f"PostgreSQL {major} | psycopg {version('psycopg')} | asyncpg {version('asyncpg')} "
            f"| events: {rows} rows")

def fatal(error):
    """What the server said, without the socket path, which is different on every machine."""
    line = str(error).strip().splitlines()[0]
    return line.rsplit("failed: ", 1)[-1]


print("server:", start_server())
print(report())


server: already running
PostgreSQL 16 | psycopg 3.3.6 | asyncpg 0.31.0 | events: 5000 rows


## Worked examples

### The four steps, one at a time

Setup did four things. This is the only notebook where they are the subject, so here they are
separately, each one asked rather than assumed.

First: is anything there? This is the question `start_server` asks first, and the answer is why it
did nothing on a machine that already had a server:


In [2]:
print("something is answering:", answering())
print("what Setup had to do:   ", start_server())


something is answering: True
what Setup had to do:    already running


Asking is a connection rather than a call to `pg_isready`, which matters: the client programs come
with the server packages, so on a machine where the server is not installed there is no `pg_isready`
to run. A driver you already installed can always ask.

Second: which server, and where does its socket live? The cluster's own settings answer both:


In [3]:
with psycopg.connect("dbname=guide") as conn:
    settings = {name: conn.execute("SELECT current_setting(%s, true)", (name,)).fetchone()[0]
                for name in ("server_version_num", "port", "unix_socket_directories")}

port = settings["port"]
directory = settings["unix_socket_directories"].split(",")[0]       # a path, different per machine
print("  server version    ", int(settings["server_version_num"]) // 10000)
print("  port              ", port)
print("  socket file       ", f".s.PGSQL.{port}")
print("  its directory is a directory that exists:", Path(directory).is_dir())


  server version     16
  port               5432
  socket file        .s.PGSQL.5432
  its directory is a directory that exists: True


The socket is an ordinary file, and its name is built from the port even though nothing here uses a
network. The directory it sits in is the one thing on this page that is different on every machine,
so it is checked rather than printed: on Colab it is `/var/run/postgresql`, and a Mac running
PostgreSQL from Homebrew puts it in `/tmp`.

Third: what is inside. A cluster holds databases, and a database holds schemas, and a schema holds
tables. Each of those is a different question:


In [4]:
with psycopg.connect("dbname=guide") as conn:
    print("databases in this cluster:",
          [row[0] for row in conn.execute(
              "SELECT datname FROM pg_database WHERE NOT datistemplate ORDER BY datname")])
    print("schemas in guide:        ",
          [row[0] for row in conn.execute(
              "SELECT nspname FROM pg_namespace WHERE nspname NOT LIKE 'pg\\_%' ORDER BY nspname")])
    print("tables in public:        ",
          [row[0] for row in conn.execute(
              "SELECT tablename FROM pg_tables WHERE schemaname = 'public' ORDER BY tablename")])


databases in this cluster: ['guide', 'postgres']
schemas in guide:         ['app', 'information_schema', 'public']
tables in public:         ['events']


Fourth: who you are. There is no password anywhere in this guide, and this is why:


In [5]:
with psycopg.connect("dbname=guide") as conn:
    who, database = conn.execute("SELECT current_user, current_database()").fetchone()
print("connected as:", "the operating system user" if who == getpass.getuser() else who)
print("to database: ", database)
print("the role is a superuser:",
      psycopg.connect("dbname=guide").execute("SELECT usesuper FROM pg_user WHERE usename = current_user").fetchone()[0])


connected as: the operating system user
to database:  guide
the role is a superuser: True


The role has the same name as the operating system user, and PostgreSQL can see which user opened
the socket, so it lets the connection through without asking for anything. That is peer
authentication, and it works only over the local socket: the same role over a network would have to
prove itself, which is **Connecting to a Hosted Server**.

### Connecting, in both drivers

The two drivers, side by side, on the same server. Neither is given a host:


In [6]:
with psycopg.connect("dbname=guide") as conn:
    print("psycopg:", conn.execute("SELECT count(*) FROM events").fetchone()[0], "events")

connection = await asyncpg.connect(database="guide")
print("asyncpg:", await connection.fetchval("SELECT count(*) FROM events"), "events")
await connection.close()


psycopg: 5000 events
asyncpg: 5000 events


Two things in there are worth naming now and are taken apart later.

The `await` is real: this notebook's kernel is already running an event loop, so `await` works at the
top level of a cell with no `asyncio.run` around it, which is what **AsyncConnection** is about.

Neither call names a host, and they find the same socket for different reasons. psycopg asks libpq,
which was compiled with a default socket directory. asyncpg has no libpq, so it looks in
`/run/postgresql`, `/var/run/postgresql`, `/tmp` and `/private/tmp` in that order, and then tries
`localhost`. Setting `PGHOST` overrides both. That list is why the same line works on Colab, on a
Mac and on a build server without being told where to look.

### Two writers

Now the thing the server is for. First the file, which is the **sqlite3, Deep Dive** behavior:


In [7]:
path = str(Path(tempfile.mkdtemp()) / "shared.db")
holder = sqlite3.connect(path, timeout=1)
waiter = sqlite3.connect(path, timeout=1)
holder.execute("CREATE TABLE writers (id INTEGER PRIMARY KEY, who TEXT)")
holder.commit()

holder.execute("BEGIN IMMEDIATE")                                   # take the write lock
holder.execute("INSERT INTO writers (who) VALUES ('first')")
try:
    waiter.execute("INSERT INTO writers (who) VALUES ('second')")
except sqlite3.OperationalError as error:
    print("the second writer:", type(error).__name__ + ":", error)

holder.rollback()
holder.close()
waiter.close()


the second writer: OperationalError: database is locked


The lock is the file, so the second writer had nothing smaller to wait for. Now the same pair against
the server, with neither committing until both have written:


In [8]:
one = psycopg.connect("dbname=guide")
two = psycopg.connect("dbname=guide")
one.execute("CREATE TABLE IF NOT EXISTS writers (id int PRIMARY KEY, who text)")
one.commit()
one.execute("TRUNCATE writers")
one.commit()

one.execute("INSERT INTO writers VALUES (1, 'first')")              # both transactions open
two.execute("INSERT INTO writers VALUES (2, 'second')")
print("both wrote before either committed")

one.commit()
two.commit()
print("rows:", one.execute("SELECT id, who FROM writers ORDER BY id").fetchall())


both wrote before either committed
rows: [(1, 'first'), (2, 'second')]


Two rows, two writers, no waiting. The lock PostgreSQL took was on the row each one wrote, so the
two never met.

They do meet on the same row, and then the second one waits rather than failing. That is worth
seeing, because "PostgreSQL does not lock" is the wrong lesson to take from the cell above:


In [9]:
one.execute("UPDATE writers SET who = 'first again' WHERE id = 1")  # holds the lock on row 1

two.execute("SET lock_timeout = '500ms'")                           # so this cell cannot hang
try:
    two.execute("UPDATE writers SET who = 'second too' WHERE id = 1")
except psycopg.errors.LockNotAvailable as error:
    print("the second writer on the same row:", fatal(error))

two.rollback()
one.rollback()
print("rows are untouched:", one.execute("SELECT id, who FROM writers ORDER BY id").fetchall())


the second writer on the same row: canceling statement due to lock timeout
rows are untouched: [(1, 'first'), (2, 'second')]


Without `lock_timeout` that second `UPDATE` would have waited for the first transaction to end,
however long that took, which is the ordinary behavior and usually the right one. The timeout is
here so a notebook cell cannot hang, and it is worth knowing about for the same reason in an
application.

### When to reach for which

| What you want | How to ask |
|---|---|
| is a server there at all | connect to `postgres` and catch `OperationalError` |
| which server, which socket | `SELECT current_setting('server_version')` and friends |
| the databases in the cluster | `SELECT datname FROM pg_database` |
| the tables in this database | `SELECT tablename FROM pg_tables WHERE schemaname = 'public'` |
| who you are connected as | `SELECT current_user, current_database()` |
| a connection, synchronously | `psycopg.connect("dbname=guide")` |
| a connection, asynchronously | `await asyncpg.connect(database="guide")` |
| a second writer not to hang | `SET lock_timeout` before the statement |

### A first program, finished

Everything above, as the thing a script would actually do on startup: check the server, say what it
is connected to, and refuse to go on if anything is missing.


In [10]:
def connect_or_explain(database="guide"):
    """Open a connection, or say in one line what is wrong with the server."""
    try:
        conn = psycopg.connect(f"dbname={database}", connect_timeout=5)
    except psycopg.OperationalError as error:
        return None, fatal(error)
    with conn:
        number, who = conn.execute(
            "SELECT current_setting('server_version_num'), current_user").fetchone()
        seen = "the operating system user" if who == getpass.getuser() else who
        return conn, f"PostgreSQL {int(number) // 10000} as {seen}"


for database in ("guide", "reports"):
    conn, message = connect_or_explain(database)
    print(f"  {database:<9} {'ok:  ' if conn else 'no:  '}{message}")


  guide     ok:  PostgreSQL 16 as the operating system user
  reports   no:  FATAL:  database "reports" does not exist


The one that worked says what it found, and the one that did not says why in a line somebody can act
on, rather than a traceback thirty lines long. Both come from the same call.

### Where each part came from

| In the program | What it relies on | The section that showed it |
|---|---|---|
| `psycopg.connect(f"dbname={database}")` | peer authentication over the local socket | The four steps |
| `connect_timeout=5` | a failure that is bounded | Common errors |
| `fatal(error)` | the server's own words, without the socket path | Setup |
| `current_setting('server_version')` | the cluster answering about itself | The four steps |
| `current_user` against the system user | why no password is needed | The four steps |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/01-a-server-of-your-own-solutions.ipynb).

**1.** Print the server's version, the database you are connected to and the role you are connected
as, in one query.


In [11]:
# your code here


**2.** List the databases in the cluster and the tables in `guide`, and say which of the two the word
"database" means in each answer.


In [12]:
# your code here


**3.** Count the `events` rows with both drivers and show that the two numbers agree.


In [13]:
# your code here


**4.** Try to connect to a database that does not exist and print only the line the server sent,
without the socket path.


In [14]:
# your code here


**5.** Open two connections, write a different row from each before either commits, and print the
table afterwards.


In [15]:
# your code here


**6.** Make two connections fight over one row, with a `lock_timeout` so your cell cannot hang, and
print what the second one was told.


In [16]:
# your code here


## Common errors

### psycopg.OperationalError: connection is bad: connection to server on socket "/tmp/nowhere/.s.PGSQL.5432" failed: No such file or directory


In [17]:
psycopg.connect("dbname=guide host=/tmp/nowhere", connect_timeout=2)


OperationalError: connection is bad: connection to server on socket "/tmp/nowhere/.s.PGSQL.5432" failed: No such file or directory
	Is the server running locally and accepting connections on that socket?

There is no socket in that directory, so there is nothing to connect to. This is the message you get
before the server is started, and it is the first one most people meet.

Read it as three facts: it looked for a **socket**, so this was a local connection and not a network
one; it looked in a **directory you can check**; and the reason is `No such file or directory`, which
is the operating system's, not PostgreSQL's. Nothing was refused, because nothing was there.

The directory comes from `PGHOST`, from the `host=` in the connection string, or from the default
libpq was compiled with:


In [18]:
print("what libpq would use with nothing set:",
      psycopg.conninfo.conninfo_to_dict(psycopg.connect("dbname=guide").info.dsn).get("host", "(its compiled default)"))
print("a server is answering there:", answering())


what libpq would use with nothing set: (its compiled default)
a server is answering there: True


### psycopg.OperationalError: connection failed: ... FATAL:  role "app" does not exist


In [19]:
try:
    psycopg.connect("dbname=guide user=app", connect_timeout=2)
except psycopg.OperationalError as error:
    print(fatal(error))


FATAL:  role "app" does not exist


The socket was there and the server answered, which is already different from the error above: this
is a refusal rather than an absence. Peer authentication only works when a role exists with the name
it is asked for, and nobody made one called `app`.

This is the error a reader meets on their own laptop rather than on Colab, because the installer
makes a role for `postgres` and not for them. Making one is the fix, and it is what Setup does for
the operating system user:


In [20]:
def role_exists(name):
    """Whether this cluster has a role of that name, which is what peer authentication checks."""
    with psycopg.connect("dbname=guide") as conn:
        return conn.execute("SELECT 1 FROM pg_roles WHERE rolname = %s", (name,)).fetchone() is not None


print("a role named for the operating system user:", role_exists(getpass.getuser()))
print("a role named app:                          ", role_exists("app"))


a role named for the operating system user: True
a role named app:                           False


### psycopg.OperationalError: connection failed: ... FATAL:  database "reports" does not exist


In [21]:
try:
    psycopg.connect("dbname=reports", connect_timeout=2)
except psycopg.OperationalError as error:
    print(fatal(error))


FATAL:  database "reports" does not exist


The server is running, the role is fine, and the database is not there. This is the one that catches
people who think of a PostgreSQL server the way they think of a SQLite file: connecting does not
create anything, and there is no mode in which it would.

Making one is a statement, and it is the one statement that cannot run inside a transaction, which
is why it needs `autocommit`. **Transactions and Errors** is where that rule comes from:


In [22]:
with psycopg.connect("dbname=postgres", autocommit=True) as conn:
    conn.execute("CREATE DATABASE reports")

with psycopg.connect("dbname=reports") as conn:
    print("now:", conn.execute("SELECT current_database()").fetchone()[0])

with psycopg.connect("dbname=postgres", autocommit=True) as conn:
    conn.execute("DROP DATABASE reports")                           # put the cluster back


now: reports


### psycopg.OperationalError: failed to resolve host 'not-a-host.invalid'


In [23]:
try:
    psycopg.connect("dbname=guide host=not-a-host.invalid", connect_timeout=2)
except psycopg.OperationalError as error:
    print(str(error).split(":")[0])                                 # the rest is your resolver's wording


failed to resolve host 'not-a-host.invalid'


A name, not a socket. The moment `host=` is something that is not a directory, this stops being a
local connection: the driver asks the operating system to turn the name into an address, and the
rest of the message is whatever your resolver said, which is worded differently on macOS and on
Linux and so is not printed here.

The distinction worth keeping is the one between the four errors in this section. Nothing at the
socket means no server. A failure to resolve means no such machine. `FATAL:  role` and
`FATAL:  database` both mean the server is running and answered you: it is your connection string
that is wrong, not the server.


## Recap

- A cluster is one running server with its own data directory, port and socket. A database is one
  named collection of tables inside it. Both get called "the database" in conversation.
- The local socket is a file in a directory, and peer authentication lets a connection through when
  a role exists with the operating system user's name, which is why nothing here needs a password.
- Neither driver needs a host for a local connection. psycopg uses libpq's compiled default;
  asyncpg looks through `/run/postgresql`, `/var/run/postgresql`, `/tmp` and `/private/tmp`, then
  `localhost`. `PGHOST` overrides both.
- `await` works at the top level of a notebook cell, because the kernel is already running a loop.
- Two writers on different rows do not wait for each other, which is what a file cannot do. Two
  writers on the same row do wait, and `lock_timeout` is how you bound that.
- Four connection failures, in increasing order of how far you got: nothing at the socket, a host
  that does not resolve, a role that does not exist, and a database that does not exist.
- Connecting never creates a database. `CREATE DATABASE` does, and it cannot run inside a
  transaction.


## What is next

The **Connecting and Executing** notebook is the connection and the cursor as separate objects:
`execute` and the three ways to get rows back, `row_factory` for rows that are dictionaries rather
than tuples, and what the `with` block around a connection actually commits when it ends.


---

[asyncpg and psycopg3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Connecting and Executing](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/02-connecting-and-executing.ipynb) &#8594;
